# 08_数据位置
**来源:** [LangChain Docs — Deep Agents Code 数据位置](https://docs.langchain.com/docs/deep-agents/code/data-locations)


## 1. 目录结构概览

Deep Agents Code 将数据存储在两大目录层级中：

```text
~/.deepagens/          # Deep Agents 特定数据
~/.agents/             # 工具无关数据（跨 AI CLI 工具共享）
<project>/             # 项目级别（Git 仓库根目录）
```

## 2. 详细目录结构

```text
~/.deepagens/
├── .state/                    # 自动管理的每机器状态
│   ├── sessions.db            # SQLite 对话检查点数据库
│   ├── history.jsonl          # 命令输入历史
│   └── ...
└── {agent}/                   # 每 Agent 目录（默认: "agent"）
    ├── AGENTS.md              # 用户对 Agent 指令的定制
    ├── skills/                # 用户级别技能
    │   └── {skill-name}/
    │       └── SKILL.md
    └── agents/                # 自定义子 Agent
        └── {subagent-name}/
            └── AGENTS.md

~/.agents/                     # 工具无关别名
└── skills/
    └── {skill-name}/
        └── SKILL.md

{project}/
├── AGENTS.md                  # 项目指令
└── .deepagents/
    ├── AGENTS.md
    ├── skills/
    └── agents/
```

## 3. 数据位置对照表

| 数据 | 位置 | 读写 | 备注 |
|------|------|------|------|
| 会话 | `~/.deepagents/.state/sessions.db` | R/W | SQLite 检查点数据库 |
| 输入历史 | `~/.deepagents/.state/history.jsonl` | R/W | JSON lines，上下箭头召回 |
| 基础指令 | 包内 `default_agent_prompt.md` | R | 不可变，随升级更新 |
| 用户定制 | `~/.deepagents/{agent}/AGENTS.md` | R/W | 追加到基础指令后 |
| 项目指令 | `.deepagents/AGENTS.md` 或 `AGENTS.md` | R/W | 两者都存在时都加载 |
| 用户技能 | `~/.deepagents/{agent}/skills/` | R/W | Agent 特定技能 |
| 共享技能 | `~/.agents/skills/` | R | 跨 CLI 工具 |
| 项目技能 | `.deepagents/skills/` | R | 项目范围 |
| 自定义子 Agent | `~/.deepagents/{agent}/agents/` | R/W | 用户定义 |
| 项目子 Agent | `.deepagents/agents/` | R | 项目定义 |

## 4. 优先级规则

**技能优先级（从低到高）：**
```text
1. ~/.deepagents/{agent}/skills/     — 用户 Deep Agents
2. ~/.agents/skills/                — 用户工具无关
3. .deepagents/skills/              — 项目 Deep Agents
4. .agents/skills/                  — 项目工具无关（最高）
```

**子 Agent 优先级（从低到高）：**
```text
1. ~/.deepagents/{agent}/agents/    — 用户级别
2. .deepagents/agents/              — 项目级别（最高）
```

**指令：** 所有来源都会被组合（不覆盖）：
```text
包基础提示（始终加载）
+ ~/.deepagents/{agent}/AGENTS.md（追加）
+ .deepagents/AGENTS.md（追加）
+ AGENTS.md（项目根目录，追加）
```

## 5. 清理操作

| 操作 | 命令 |
|------|------|
| 重置所有数据 | `rm -rf ~/.deepagents` |
| 清除会话 | `rm ~/.deepagents/.state/sessions.db*` |
| 清除输入历史 | `rm ~/.deepagents/.state/history.jsonl` |
| 清除存储的 API 密钥 | `rm ~/.deepagents/.state/auth.json` |
| 清除 MCP OAuth 令牌 | `rm -rf ~/.deepagents/.state/mcp-tokens` |
| 重新运行首次引导 | `rm ~/.deepagents/.state/onboarding_complete` |
| 重置 Agent 指令 | `dcode agents reset --agent {name}` |
| 删除技能 | `rm -rf ~/.deepagents/{agent}/skills/{skill-name}` |

## 6. `.deepagents` vs `.agents`

| 目录 | 用途 | 使用场景 |
|------|------|---------|
| `.deepagents/` | Deep Agents Code 特定 | 使用 Deep Agents 特定特性的技能和配置 |
| `.agents/` | 工具无关 | 希望跨不同 AI CLI 工具共享的技能 |

> 使用 `.agents/skills/` 存放可与任意 AI 编码助手协作的技能；
> 使用 `.deepagents/skills/` 存放依赖 Deep Agents 特定工具或约定的技能。

## 参考

- [配置文档](https://docs.langchain.com/docs/deep-agents/code/configuration)
- [记忆与技能文档](https://docs.langchain.com/docs/deep-agents/code/memory-and-skills)
- [子 Agent 文档](https://docs.langchain.com/docs/deep-agents/code/subagents)